# Fase 3: Separazione dei Dati, Imputazione e Bilanciamento (SMOTE/PCA)

In questa fase si suddivide il dataset in **Training Set (60%)**, **Validation Set (20%)** e **Test Set (20%)** in modo stratificato rispetto al target clinico `TenYearCHD`.

Al fine di evitare qualsiasi forma di **data leakage**, l'imputazione KNN, lo scaling (RobustScaler), il bilanciamento delle classi (SMOTE) e la riduzione di dimensionalità (PCA) vengono calcolati **esclusivamente sul Training Set** e applicati in cascata sui set di Validation e Test.

In [1]:
# Importiamo i moduli necessari per la modellizzazione e la manipolazione dei dati
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE

In [2]:
# Caricamento del dataset pre-imputazione
dati_pre_imputazione = pd.read_csv("../Dataset/Clean/framingham_clean_pre_imputation.csv")

# Dividiamo le feature esplicative (X) dalla variabile target (Y)
X = dati_pre_imputazione.drop(columns=['TenYearCHD'])
Y = dati_pre_imputazione['TenYearCHD']

# Creiamo il Test Set (20% del totale)
X_train_val_grezzo, X_test_grezzo, Y_train_val, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# Suddivisione della parte restante in Train Set (60%) e Validation Set (20%)
X_train_grezzo, X_val_grezzo, Y_train, Y_val = train_test_split(
    X_train_val_grezzo, Y_train_val, test_size=0.25, random_state=42, stratify=Y_train_val
)

print(f"Suddivisione completata in modo corretto:")
print(f"Training Set: {X_train_grezzo.shape} pazienti")
print(f"Validation Set: {X_val_grezzo.shape} pazienti")
print(f"Test Set: {X_test_grezzo.shape} pazienti")

Suddivisione completata in modo corretto:
Training Set: (2542, 15) pazienti
Validation Set: (848, 15) pazienti
Test Set: (848, 15) pazienti


In [3]:
# Definizione della cartella base di salvataggio per i dati di training
CARTELLA_TRAINING = "../Training"

# Definizione dei due rami principali: senza scaling ('Clean') e standardizzato ('Normalized')
modalita_scala = ["Clean", "Normalized"]

# Esecuzione di un ciclo esplicito e commentato per gestire la pipeline di pre-elaborazione senza leakage
for modalita in modalita_scala:
    print(f"\n==================== PIPELINE RAMO: {modalita} ====================")
    
    # 1. Imputazione dei dati mancanti: fittiamo KNNImputer SOLO sui dati di train ed imputiamo val e test
    imputatore_knn = KNNImputer(n_neighbors=5)
    X_train_imp = pd.DataFrame(
        imputatore_knn.fit_transform(X_train_grezzo), 
        columns=X_train_grezzo.columns, 
        index=X_train_grezzo.index
    )
    X_val_imp = pd.DataFrame(
        imputatore_knn.transform(X_val_grezzo), 
        columns=X_val_grezzo.columns, 
        index=X_val_grezzo.index
    )
    X_test_imp = pd.DataFrame(
        imputatore_knn.transform(X_test_grezzo), 
        columns=X_test_grezzo.columns, 
        index=X_test_grezzo.index
    )
    
    # 2. Se previsto, applichiamo lo scaling (RobustScaler) fittando SOLO sul Training Set
    if modalita == "Normalized":
        scaler_robusto = RobustScaler()
        colonne_da_scalare = ['totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
        
        X_train_lavoro = X_train_imp.copy()
        X_val_lavoro = X_val_imp.copy()
        X_test_lavoro = X_test_imp.copy()
        
        X_train_lavoro[colonne_da_scalare] = scaler_robusto.fit_transform(X_train_imp[colonne_da_scalare])
        X_val_lavoro[colonne_da_scalare] = scaler_robusto.transform(X_val_imp[colonne_da_scalare])
        X_test_lavoro[colonne_da_scalare] = scaler_robusto.transform(X_test_imp[colonne_da_scalare])
    else:
        X_train_lavoro = X_train_imp.copy()
        X_val_lavoro = X_val_imp.copy()
        X_test_lavoro = X_test_imp.copy()
        
    # --- SOTTO-RAMO A: NORMAL (Dati originali sbilanciati) ---
    dir_normal = os.path.join(CARTELLA_TRAINING, modalita, "Normal")
    os.makedirs(dir_normal, exist_ok=True)
    X_train_lavoro.to_csv(os.path.join(dir_normal, "X_train.csv"), index=False)
    X_val_lavoro.to_csv(os.path.join(dir_normal, "X_val.csv"), index=False)
    X_test_lavoro.to_csv(os.path.join(dir_normal, "X_test.csv"), index=False)
    Y_train.to_csv(os.path.join(dir_normal, "Y_train.csv"), index=False, header=True)
    Y_val.to_csv(os.path.join(dir_normal, "Y_val.csv"), index=False, header=True)
    Y_test.to_csv(os.path.join(dir_normal, "Y_test.csv"), index=False, header=True)
    print(f"✅ Salvato subset Normal in: {dir_normal}")
    
    # --- SOTTO-RAMO B: AUGMENTED (Bilanciamento classi tramite SMOTE) ---
    # SMOTE viene fittato ed applicato SOLO al Training Set. Val e Test rimangono sbilanciati ed intatti
    smote_algoritmo = SMOTE(random_state=42)
    X_train_bilanciato, Y_train_bilanciato = smote_algoritmo.fit_resample(X_train_lavoro, Y_train)
    
    dir_aug = os.path.join(CARTELLA_TRAINING, modalita, "Aug")
    os.makedirs(dir_aug, exist_ok=True)
    X_train_bilanciato.to_csv(os.path.join(dir_aug, "X_train.csv"), index=False)
    X_val_lavoro.to_csv(os.path.join(dir_aug, "X_val.csv"), index=False)
    X_test_lavoro.to_csv(os.path.join(dir_aug, "X_test.csv"), index=False)
    Y_train_bilanciato.to_csv(os.path.join(dir_aug, "Y_train.csv"), index=False, header=True)
    Y_val.to_csv(os.path.join(dir_aug, "Y_val.csv"), index=False, header=True)
    Y_test.to_csv(os.path.join(dir_aug, "Y_test.csv"), index=False, header=True)
    print(f"✅ Salvato subset Aug in: {dir_aug}")
    
    # --- SOTTO-RAMO C: PCA (Riduzione della dimensionalità a 6 componenti) ---
    riduttore_pca = PCA(n_components=6, random_state=42)
    X_train_pca = riduttore_pca.fit_transform(X_train_lavoro)
    X_val_pca = riduttore_pca.transform(X_val_lavoro)
    X_test_pca = riduttore_pca.transform(X_test_lavoro)
    
    nomi_componenti = [f'PC{i+1}' for i in range(X_train_pca.shape[1])]
    df_train_pca = pd.DataFrame(X_train_pca, columns=nomi_componenti, index=X_train_lavoro.index)
    df_val_pca = pd.DataFrame(X_val_pca, columns=nomi_componenti, index=X_val_lavoro.index)
    df_test_pca = pd.DataFrame(X_test_pca, columns=nomi_componenti, index=X_test_lavoro.index)
    
    dir_pca = os.path.join(CARTELLA_TRAINING, modalita, "Pca")
    os.makedirs(dir_pca, exist_ok=True)
    df_train_pca.to_csv(os.path.join(dir_pca, "X_train.csv"), index=False)
    df_val_pca.to_csv(os.path.join(dir_pca, "X_val.csv"), index=False)
    df_test_pca.to_csv(os.path.join(dir_pca, "X_test.csv"), index=False)
    Y_train.to_csv(os.path.join(dir_pca, "Y_train.csv"), index=False, header=True)
    Y_val.to_csv(os.path.join(dir_pca, "Y_val.csv"), index=False, header=True)
    Y_test.to_csv(os.path.join(dir_pca, "Y_test.csv"), index=False, header=True)
    print(f"✅ Salvato subset PCA in: {dir_pca}")
    
    # --- SOTTO-RAMO D: AUGMENTED + PCA (PCA su dati bilanciati con SMOTE) ---
    riduttore_pca_aug = PCA(n_components=6, random_state=42)
    X_train_aug_pca = riduttore_pca_aug.fit_transform(X_train_bilanciato)
    X_val_aug_pca = riduttore_pca_aug.transform(X_val_lavoro)
    X_test_aug_pca = riduttore_pca_aug.transform(X_test_lavoro)
    
    df_train_aug_pca = pd.DataFrame(X_train_aug_pca, columns=nomi_componenti)
    df_val_aug_pca = pd.DataFrame(X_val_aug_pca, columns=nomi_componenti)
    df_test_aug_pca = pd.DataFrame(X_test_aug_pca, columns=nomi_componenti)
    
    dir_aug_pca = os.path.join(CARTELLA_TRAINING, modalita, "Aug+Pca")
    os.makedirs(dir_aug_pca, exist_ok=True)
    df_train_aug_pca.to_csv(os.path.join(dir_aug_pca, "X_train.csv"), index=False)
    df_val_aug_pca.to_csv(os.path.join(dir_aug_pca, "X_val.csv"), index=False)
    df_test_aug_pca.to_csv(os.path.join(dir_aug_pca, "X_test.csv"), index=False)
    Y_train_bilanciato.to_csv(os.path.join(dir_aug_pca, "Y_train.csv"), index=False, header=True)
    Y_val.to_csv(os.path.join(dir_aug_pca, "Y_val.csv"), index=False, header=True)
    Y_test.to_csv(os.path.join(dir_aug_pca, "Y_test.csv"), index=False, header=True)
    print(f"✅ Salvato subset Aug+PCA in: {dir_aug_pca}")

print("\n🎉 Tutti i dataset per le 8 combinazioni sono stati generati e salvati correttamente senza data leakage!")


==================== PIPELINE RAMO: Clean ====================
✅ Salvato subset Normal in: ../Training/Clean/Normal
✅ Salvato subset Aug in: ../Training/Clean/Aug
✅ Salvato subset PCA in: ../Training/Clean/Pca
✅ Salvato subset Aug+PCA in: ../Training/Clean/Aug+Pca

==================== PIPELINE RAMO: Normalized ====================
✅ Salvato subset Normal in: ../Training/Normalized/Normal
✅ Salvato subset Aug in: ../Training/Normalized/Aug
✅ Salvato subset PCA in: ../Training/Normalized/Pca
✅ Salvato subset Aug+PCA in: ../Training/Normalized/Aug+Pca

🎉 Tutti i dataset per le 8 combinazioni sono stati generati e salvati correttamente senza data leakage!
